![Modernization Narrative Walkthrough — governed pipeline and evidence lifecycle: provenance, contracts, transformations, subject-grouped splits, validation evidence, and limitations](../docs/assets/ecg-notebook-narrative-walkthrough-banner.png)

# Step 1 — ECG pipeline modernization: a supported narrative walkthrough — v1.10

Welcome! This notebook tells the story of how a historical educational ECG notebook project became a reproducible data-engineering case study. You will follow source provenance, configuration contracts, annotation mapping, window extraction, subject-aware splitting, validation-only evaluation, and auditable run evidence without reimplementing package logic.

A first guided read typically takes about **10–15 minutes**. Once Step 0 artifacts exist, running all cells is usually **under a minute**; reruns are similar, although scanning many local run manifests may take longer. These are orientation ranges, not guarantees.

**Jump to:** [modernization context](#1-repository-purpose-and-modernization-context) · [configuration workflow](#3-configuration-driven-workflow) · [subject-aware splitting](#7-subject-aware-splitting-and-split-diagnostics) · [run evidence](#9-reproducibility-evidence-run-manifests-and-artifact-lineage) · [limitations](#11-portfolio-interpretation-and-limitations) · [version appendix](#appendix-version-history-and-change-log)

Implementation remains in the repository package, configuration, CLI, and tests; the [pipeline design](../docs/pipeline-design.md) and [pipeline orchestration](../docs/pipeline-orchestration.md) documents define those maintained contracts.

<div role="note" aria-label="Research and educational use boundary" style="border: 3px solid #007c91; border-left-width: 8px; border-radius: 6px; background: #eefbff; color: #102a43; padding: 1rem 1.25rem; margin: 1rem 0; line-height: 1.5;"><strong style="display: block; font-size: 1.15rem; margin-bottom: 0.35rem;">Research and educational use boundary</strong>This repository is not medical software, clinical software, or production ML. It makes no clinical, diagnostic, deployment, or medical-utility claims. Validation evidence is not benchmark evidence.</div>


## 0. Confirm the supported local checkout

Run this cell first. It confirms that the notebook is using the repository checkout and locked project kernel, then keeps notebook 00 generated state in place for the prerequisite checks below. No data or artifacts are copied. Optional web-hosted execution is deferred to [Issue #200](https://github.com/Jared-Godar/ecg_anomaly_detection/issues/200).


> **CODE CELL FUNCTION**
>
> Confirm the supported local checkout and project kernel before reading Step 0 state.


In [ ]:
# CODE CELL FUNCTION:
# Confirm the supported local checkout and project kernel before downstream work.
#
# COMMENT MAP:
# - Locate the repository root even when Jupyter starts inside notebooks/.
# - Require the locked project kernel selected during local setup.
# - Reuse ignored Step 0 artifacts in place without copying generated state.
# - Keep the protected test partition unopened.

from __future__ import annotations

import os
import sys
from pathlib import Path

# Jupyter may start with notebooks/ as its working directory, so walk upward to
# find the two stable repository markers used throughout the curated workflow.
repository_root = None
# Inspect candidates from nearest to farthest so this checkout wins deterministically.
for candidate in (Path.cwd(), *Path.cwd().parents):
    # Requiring both markers avoids adopting an unrelated Python-project ancestor.
    if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
        repository_root = candidate.resolve()
        break
# Continuing outside the checkout would make every relative artifact path ambiguous.
if repository_root is None:
    raise RuntimeError("Repository root not found. Open this notebook from the local project checkout.")

# The supported workflow uses the checkout-local environment created by uv. Compare
# the interpreter path lexically: uv creates the venv with the interpreter symlinked
# to a base Python outside .venv, so resolving it would falsely reject the correctly
# selected kernel. This gate still gives a direct remediation before a later import error.
expected_kernel_directory = (repository_root / ".venv").absolute()
# Keep the displayed executable path without resolving through the venv symlink.
kernel_executable = Path(sys.executable).absolute()
# Stop before imports when the editor selected a global or unrelated kernel.
if not kernel_executable.is_relative_to(expected_kernel_directory):
    raise RuntimeError(
        "Select the checkout-local .venv Python kernel before continuing.\n"
        f"Expected beneath: {expected_kernel_directory}\n"
        f"Current kernel: {kernel_executable}"
    )

# Anchor all later repository-relative paths to the verified local checkout.
os.chdir(repository_root)
# Publish a simple gate that later review/debug cells can inspect in memory.
CONTINUITY_READY = True
# Keep the local no-copy and protected-partition contract visible to the user.
NOTEBOOK_CONTINUITY = {
    "status": "local_checkout_ready",
    "execution_environment": "local checkout",
    "action": "reuse_generated_state_in_place",
    "repository_root": str(repository_root),
    "generated_state_copied": False,
    "protected_test_partition_opened": False,
}
print("[continuity] Local checkout and project kernel confirmed; Step 0 state remains in place.", flush=True)

NOTEBOOK_CONTINUITY


<div role="alert" aria-label="Step 0 prerequisite" style="border: 3px solid #b42318; border-left-width: 8px; border-radius: 6px; background: #fff4f2; color: #5c160f; padding: 1rem 1.25rem; margin: 1rem 0; line-height: 1.5;"><strong style="display: block; font-size: 1.15rem; margin-bottom: 0.35rem;">Prerequisite required: complete Step 0</strong>Complete <a href="./00-environment-setup-and-artifact-generation.ipynb">00-environment-setup-and-artifact-generation.ipynb</a> before using generated local artifacts. This notebook explains the governed modernization lifecycle; it does not install the environment, clean generated outputs, acquire data, or run the full pipeline.</div>


> **CODE CELL FUNCTION**
>
> Run a compact Step 0 readiness check before continuing. The cell uses only Python standard-library modules and fails with a clear remediation message if the local checkout has not completed Step 0.

In [ ]:
# CODE CELL FUNCTION:
# Enforce Step 0 completion before the narrative walkthrough inspects generated state.
#
# COMMENT MAP:
# - Read the machine-readable Step 0 status written by notebook 00.
# - Require actual model-ready artifacts, not just a completed notebook run.
# - Fail fast when Step 0 stopped at PIPE-006 or any other pipeline error.
# - Keep this cell read-only so it verifies state without mutating local artifacts.
# - Preserve reviewer UX by making the missing dependency explicit at the top of Step 1.

from __future__ import annotations

import json
from pathlib import Path
from typing import Any

# Anchor every relative path in this cell to the kernel's working directory, which
# must be the repository root (Step 0's own diagnostics cell verifies this).
ROOT = Path.cwd()
# The cell-to-cell contract file Step 0's pipeline-execution cell writes.
STEP0_STATUS_PATH = ROOT / "notebooks" / "local" / "step0-pipeline-status.json"


def load_json_object(path: Path) -> dict[str, Any]:
    """Load a JSON object from disk and reject missing or malformed status artifacts."""
    # A missing status file means Step 0's pipeline-execution cell never ran.
    if not path.is_file():
        raise RuntimeError(
            "Step 1 requires Step 0 to complete first, but the Step 0 status file is missing.\n"
            f"Expected: {path}\n"
            "Run notebooks/00-environment-setup-and-artifact-generation.ipynb from the top."
        )
    # A status file that fails to parse means it was corrupted or hand-edited.
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        raise RuntimeError(f"Step 0 status file is invalid JSON: {path}") from exc
    # Only an object matches this contract's expected shape (status/reason/... keys).
    if not isinstance(payload, dict):
        raise RuntimeError(f"Step 0 status file must contain a JSON object: {path}")
    return payload


def newest_dataset_index(root: Path) -> Path:
    """Return the newest generated dataset index produced by strict Step 0."""
    candidates = sorted(
        (root / "data" / "processed" / "runs").glob("*/dataset-index.json"),
        key=lambda path: path.stat().st_mtime,
    )
    # No dataset index anywhere means Step 0 never reached the indexing stage,
    # regardless of what its own status file claims.
    if not candidates:
        raise RuntimeError(
            "Step 0 has not produced data/processed/runs/<run-id>/dataset-index.json. "
            "Step 1 will not continue because the public workflow is not artifact-ready."
        )
    return candidates[-1]


def require_file(path: Path, label: str) -> Path:
    """Require one downstream artifact without attempting repair from Step 1."""
    # Step 1 is read-only with respect to Step 0's outputs: it verifies, never repairs.
    if not path.is_file():
        raise RuntimeError(f"Missing required Step 0 artifact for Step 1: {label}: {path}")
    return path


# Load Step 0's own verdict first; every check below assumes it succeeded.
status = load_json_object(STEP0_STATUS_PATH)
# A blocked/failed status must raise immediately rather than proceeding to the
# artifact-existence checks below, which assume a completed run.
if status.get("status") != "complete":
    raise RuntimeError(
        "Step 1 requires Step 0 to complete artifact generation. Step 0 did not complete.\n"
        f"Status: {status.get('status')}\n"
        f"Reason: {status.get('reason')}\n"
        f"Message: {status.get('message')}\n"
        f"Log file: {status.get('log_file')}"
    )

# A "complete" status alone is not sufficient proof; re-derive and re-verify the
# actual generated artifacts independently of what the status file claims.
index_path = newest_dataset_index(ROOT)
# The dataset index's own parent directory name is the run ID that produced it.
run_id = index_path.parent.name
# The remaining required artifacts live in this run's artifacts/runs/<run-id>/ tree.
run_root = ROOT / "artifacts" / "runs" / run_id
# The exact set of generated files this narrative walkthrough depends on, labeled
# for the readable missing-artifact report require_file raises on demand.
verified = {
    "dataset_index": require_file(index_path, "dataset index"),
    "split_manifest": require_file(run_root / "split.json", "split manifest"),
    "split_quality": require_file(run_root / "split_quality_summary.json", "split quality summary"),
    "run_manifest": require_file(run_root / "run-manifest.json", "run manifest"),
}

{
    "step_0_status": status.get("status"),
    "run_id": run_id,
    "verified_artifacts": {label: str(path.relative_to(ROOT)) for label, path in verified.items()},
    "narrative_walkthrough_ready": True,
    "test_partition_opened": False,
}


## 1. Repository purpose and modernization context

The supported workflow replaces notebook-bound transformations with testable package boundaries. The notebook reads those contracts; it does not reimplement them. The diagram below is a modern repository-owned asset, not an image from the historical archive. Its contracts are described in the [pipeline design document](../docs/pipeline-design.md).

![Flow diagram of the modern pipeline and evidence lineage. Versioned configuration, tested package contracts, and CLI orchestration drive acquisition and inventory, validation and annotation mapping, labeled-window extraction, subject-aware splitting and dataset indexing, training-only fitting, and validation-only evaluation. Generated evidence feeds an auditable run manifest. The protected test partition is indexed but remains unopened and unreported.](../reports/figures/modern-pipeline-lineage.svg)

*Modern pipeline and evidence lineage: a local research workflow from versioned controls through validation-only evidence and an auditable run manifest, with the protected test partition unopened.* — generated from [`modern-pipeline-lineage.dot`](../docs/diagrams/src/modern-pipeline-lineage.dot) via Graphviz (see the [diagram design specification](../docs/diagrams/design-spec.md)).

> **CODE CELL FUNCTION**
>
> Run the next walkthrough cell. This cell is package-backed and uses repository-relative paths.

In [ ]:
# CODE CELL FUNCTION:
# Load package-backed configuration readers and establish repository-relative paths.
#
# COMMENT MAP:
# - This cell imports repository package modules, standard-library helpers, and shared progress UX.
# - It does not read generated datasets or model artifacts.
# - It configures one stdout reporter for the later variable-duration evidence scan.
# - It creates the shared ``paths`` and ``root`` variables used by later cells.
# - If this cell fails, the environment was not launched through the project setup
#   expected by Step 0, typically ``uv run --group notebooks jupyter lab``.

from dataclasses import asdict
import json
from pathlib import Path
import sys

from ecg_anomaly_detection.benchmark_policy import load_benchmark_policy
from ecg_anomaly_detection.config import RepositoryPaths, load_dataset_config
from ecg_anomaly_detection.evaluation import load_evaluation_config
from ecg_anomaly_detection.labels import load_annotation_mapping
from ecg_anomaly_detection.progress import ProgressReporter
from ecg_anomaly_detection.splitting import load_split_config
from ecg_anomaly_detection.training import load_training_config
from ecg_anomaly_detection.windows import load_window_config

# Discovering paths through the package keeps notebook path handling aligned
# with the CLI and tests instead of reimplementing repository layout logic here.
paths = RepositoryPaths.discover()
# Every later cell in this notebook anchors its own paths off this one root.
root = paths.root
# Reuse the package's immediately flushed, observational-only reporter instead
# of creating notebook-specific timing or output behavior.
NARRATIVE_PROGRESS = ProgressReporter(stream=sys.stdout)

# Return a small audit object instead of printing prose so execution output is
# compact, deterministic, and easy to inspect in saved notebooks.
{
    "repository_root": str(root),
    "package_backed": True,
}


## 2. Historical archive boundary

The notebooks in the [preserved 2022 archive](../archive/original_2022/README.md) are historical artifacts. They are unsupported, may contain stale errors and duplicated exploratory logic, and are not executed by this walkthrough. Their saved metrics remain [historical results with known evaluation limitations](../docs/historical-results.md); they do not demonstrate generalization to unseen patients. No historical image is reused here.

> **CODE CELL FUNCTION**
>
> Run the next walkthrough cell. This cell is package-backed and uses repository-relative paths.

In [ ]:
# CODE CELL FUNCTION:
# Confirm the historical archive boundary without executing archived notebooks.
#
# COMMENT MAP:
# - This cell checks only whether the preserved historical archive directory exists.
# - It does not import, execute, rewrite, or validate unsupported historical notebooks.
# - The output supports the narrative distinction between preserved history and
#   the supported modern package-backed workflow.

archive = root / "archive" / "original_2022"

{
    "archive_exists": archive.is_dir(),
    "archive_role": "preserved and unsupported",
}


## 3. Configuration-driven workflow

Each stage validates a committed TOML contract before touching data. Loading these contracts is deterministic and data-independent.

> **CODE CELL FUNCTION**
>
> Run the next walkthrough cell. This cell is package-backed and uses repository-relative paths.

In [ ]:
# CODE CELL FUNCTION:
# Load the committed configuration contracts that define the supported workflow.
#
# COMMENT MAP:
# - Each loader validates a versioned TOML file from ``configs/``.
# - The cell is deterministic and data-independent; it does not require local MIT-BIH files.
# - The output reports contract identity rather than model performance.

# Source dataset identity, provenance URLs, and expected record inventory.
dataset = load_dataset_config(paths.configs / "mitdb-v1.0.0.toml")
# Closed-world annotation-symbol-to-binary-target mapping (labels.py).
mapping = load_annotation_mapping(paths.configs / "annotation-map-v1.toml")
# Beat-centered window extraction geometry (pre/post margins, channel selection).
windowing = load_window_config(paths.configs / "windowing-v1.toml")
# Record/subject-grouped train/val/test partition strategy and quality thresholds.
splitting = load_split_config(paths.configs / "splitting-v2.toml")
# Baseline estimator hyperparameters and training-only data source.
training = load_training_config(paths.configs / "training-baseline-v1.toml")
# Validation-only evaluation metrics and reporting configuration.
evaluation = load_evaluation_config(paths.configs / "evaluation-baseline-v1.toml")

{
    "dataset": f"{dataset.slug}@{dataset.version}",
    "records": len(dataset.record_ids),
    "mapping": f"{mapping.name}@{mapping.version}",
    "windowing": f"{windowing.name}@{windowing.version}",
    "splitting": f"{splitting.name}@{splitting.version}",
    "training": f"{training.name}@{training.version}",
    "evaluation": f"{evaluation.name}@{evaluation.version}",
}


## 4. Acquisition and source validation

The acquisition stage retrieves the configured public MIT-BIH release into the ignored raw-data zone, checks the repository-reviewed expected file inventory, and records acquisition and SHA-256 evidence under the repository's [data-provenance contract](../docs/data-provenance.md). This notebook deliberately does not download the complete dataset during routine validation. A local full run uses the supported command shown later.

> **CODE CELL FUNCTION**
>
> Run the next walkthrough cell. This cell is package-backed and uses repository-relative paths.

In [ ]:
# CODE CELL FUNCTION:
# Summarize acquisition contract metadata without downloading the dataset.
#
# COMMENT MAP:
# - This cell reads committed dataset configuration already loaded above.
# - It does not call the acquisition CLI, access the network, or inspect raw ECG records.
# - The ``raw_data_committed`` value documents the repository boundary: raw data stays
#   out of Git and is generated locally through Step 0.

{
    "source": dataset.source_url,
    "sample_rate_hz": dataset.sample_rate_hz,
    "required_files": len(dataset.expected_files),
    "reviewed_source_digests": len(dataset.expected_source_files),
    "raw_data_committed": False,
}


## 5. Annotation mapping

The package applies a [closed-world, versioned annotation mapping](../docs/annotation-mapping.md). Every source symbol must be assigned to a target or explicitly excluded; unknown symbols fail closed. Per-record JSON reports preserve inclusion, exclusion, and target counts.

> **CODE CELL FUNCTION**
>
> Run the next walkthrough cell. This cell is package-backed and uses repository-relative paths.

In [ ]:
# CODE CELL FUNCTION:
# Show the closed-world annotation mapping contract.
#
# COMMENT MAP:
# - The mapping object comes from the committed TOML contract loaded earlier.
# - Unknown symbols are intentionally fail-closed in the package layer.
# - This cell reports target/exclusion policy only; it does not read annotation files.

{
    "unknown_symbol_policy": mapping.unknown_symbol_policy,
    "targets": {rule.name: list(rule.symbols) for rule in mapping.targets},
    "explicitly_excluded_symbols": list(mapping.excluded_symbols),
}


## 6. Window extraction

The package follows the documented [window-extraction contract](../docs/window-extraction.md): it converts configured time geometry into samples, selects the configured channel, excludes incomplete boundary windows, and retains record ID, annotation location, source symbol, and target lineage. It also reports adjacent overlap rather than hiding it.

> **CODE CELL FUNCTION**
>
> Run the next walkthrough cell. This cell is package-backed and uses repository-relative paths.

In [ ]:
# CODE CELL FUNCTION:
# Display the configured beat-window extraction geometry and lineage fields.
#
# COMMENT MAP:
# - This reports the committed windowing contract; it does not extract windows.
# - The channel setting is intentionally visible because PIPE-006 concerns channel
#   identity consistency across records.
# - Explicit record exclusions explain why 48 configured source records produce
#   processed artifacts for 46 records under the committed windowing contract.
# - The lineage field list documents what downstream artifacts must preserve.

{
    "pre_seconds": windowing.pre_seconds,
    "post_seconds": windowing.post_seconds,
    "channel_name": windowing.channel_name,
    "channel_index": windowing.channel_index,
    "excluded_record_ids": list(windowing.exclude_record_ids),
    "boundary_policy": windowing.boundary_policy,
    "lineage_fields": [
        "record_id",
        "center_sample_index",
        "source_symbol",
        "target_value",
    ],
}


## 7. Subject-aware splitting and split diagnostics

The [record-grouped splitting contract](../docs/record-grouped-splitting.md) maps records to subjects and assigns complete subjects to deterministic train, validation, and protected test partitions. Package assertions reject subject or record crossover. A generated split-quality summary records disjointness, partition counts, class coverage, ratio deviations, and configured warning/failure thresholds.

Partition assignment is deliberately seeded: the split config fixes a shuffle `seed` (2022 in [`configs/splitting-v2.toml`](../configs/splitting-v2.toml)), so the subject shuffle — and therefore train/validation/test membership — comes out identical on every rerun, on any machine. That stability is what keeps split diagnostics and downstream validation results comparable across runs. The next cell surfaces the seed alongside the other split-contract fields.

> **CODE CELL FUNCTION**
>
> Run the next walkthrough cell. This cell is package-backed and uses repository-relative paths.

In [ ]:
# CODE CELL FUNCTION:
# Summarize the subject-aware split contract without opening generated shards.
#
# COMMENT MAP:
# - The split contract is committed configuration, not generated data.
# - Subject grouping is the relevant modernization behavior; records should not
#   cross train/validation/test partitions.
# - This output supports evaluation-governance discussion without computing metrics.

{
    "schema_version": splitting.schema_version,
    "strategy": splitting.strategy,
    "seed": splitting.seed,
    "ratios": {
        "train": splitting.train_ratio,
        "validation": splitting.validation_ratio,
        "test": splitting.test_ratio,
    },
    "record_subject_mappings": len(splitting.record_subjects),
}


## 8. Baseline training and validation-only evaluation

Training resolves only training shards from the model-ready index. Evaluation then loads the frozen model and resolves only validation shards, verifying content digests before scoring under the [evaluation policy](../docs/evaluation-policy.md). Validation metrics are pipeline evidence—not benchmark evidence, protected-test evidence, or proof of generalization.

Training is seeded for the same reason the split is: the training config fixes `seed = 2022` (in [`configs/training-baseline-v1.toml`](../configs/training-baseline-v1.toml)) for the baseline's random feature projection, so refitting from identical artifacts produces an identical model. The contract summary below reports that seed.

> **CODE CELL FUNCTION**
>
> Run the next walkthrough cell. This cell is package-backed and uses repository-relative paths.

In [ ]:
# CODE CELL FUNCTION:
# Report the baseline training/evaluation contract and protected-test boundary.
#
# COMMENT MAP:
# - This cell does not train, score, or load a model.
# - It documents that routine evaluation is validation-only.
# - The protected test partition remains unopened by this walkthrough.

{
    "estimator": training.estimator,
    "training_seed": training.seed,
    "training_partition": "train",
    "evaluation_partition": evaluation.partition,
    "test_partition_accessed": False,
}


## 9. Reproducibility evidence, run manifests, and artifact lineage

A supported full run writes [reproducibility evidence](../docs/reproducibility-evidence.md) beneath an ignored UUID-scoped run directory. The final [run manifest](../docs/run-manifests.md) links environment, runtime, resource, configuration, report, artifact, and digest records. The next cell reads only existing JSON evidence; when local artifacts are absent, it returns an explicit status and continues cleanly.

> **CODE CELL FUNCTION**
>
> Run the next walkthrough cell. This cell is package-backed and uses repository-relative paths.

In [ ]:
# CODE CELL FUNCTION:
# Inspect optional local run-manifest evidence written by a completed Step 0 run.
#
# COMMENT MAP:
# - This cell reads JSON evidence only if it already exists under ignored artifacts.
# - It does not run the pipeline, clean state, generate metrics, or open protected test data.
# - It reports one bounded progress pair because many ignored local runs can lengthen discovery.
# - Missing evidence is an expected state when Step 0 is blocked by PIPE-006 or has
#   not been run yet, so the cell returns status instead of failing.

run_root = paths.artifacts / "runs"

# Most checkouts finish this scan almost immediately, but the expectation names
# the two local factors that can make a large ignored run history take longer.
with NARRATIVE_PROGRESS.stage(
    "inspect optional local run evidence",
    1,
    1,
    detail="usually seconds; number of local runs and disk speed vary",
) as evidence_stage:
    # Sorting by modification time makes "latest" reflect the most recent local run
    # rather than lexicographic UUID order.
    run_manifests = (
        sorted(
            run_root.glob("*/run-manifest.json"),
            key=lambda path: path.stat().st_mtime,
        )
        if run_root.is_dir()
        else []
    )

    # Local run evidence is genuinely optional here (see the cell's own COMMENT MAP
    # above); report a status object either way rather than raising on absence.
    if run_manifests:
        latest_manifest_path = run_manifests[-1]
        latest_manifest = json.loads(latest_manifest_path.read_text(encoding="utf-8"))

        evidence_status = {
            "status": "local evidence available",
            "manifest": str(latest_manifest_path.relative_to(root)),
            "schema_version": latest_manifest.get("schema_version"),
            "top_level_fields": sorted(latest_manifest),
        }
        # The completion detail names evidence presence without duplicating the
        # manifest payload or turning progress output into a second results surface.
        evidence_stage.detail("latest local run manifest loaded")
    else:
        evidence_status = {
            "status": "no local run evidence available",
            "expected_location": "artifacts/runs/<run-id>/run-manifest.json",
            "likely_reason": (
                "Step 0 has not completed artifact generation, or it ended in a controlled "
                "blocked state such as PIPE-006."
            ),
        }
        # Absence is expected and non-fatal, so the completion line reports it
        # plainly instead of suggesting that evidence generation was attempted.
        evidence_stage.detail("no local run manifest present; no artifacts written")

evidence_status


Run the complete supported workflow from the repository root when local data acquisition is intended:

```fish
uv run ecg-data run-pipeline \
  --repository-root . \
  --dataset-config configs/mitdb-v1.0.0.toml \
  --mapping-config configs/annotation-map-v1.toml \
  --window-config configs/windowing-v1.toml \
  --split-config configs/splitting-v2.toml \
  --training-config configs/training-baseline-v1.toml \
  --evaluation-config configs/evaluation-baseline-v1.toml
```

## 10. Benchmark governance boundary

<div role="alert" aria-label="Protected test and benchmark boundary" style="border: 3px solid #b42318; border-left-width: 8px; border-radius: 6px; background: #fff4f2; color: #5c160f; padding: 1rem 1.25rem; margin: 1rem 0; line-height: 1.5;"><strong style="display: block; font-size: 1.15rem; margin-bottom: 0.35rem;">Protected test boundary</strong>The held-out test partition is intentionally unavailable for routine evaluation. The <a href="../docs/benchmark-governance.md">benchmark policy</a> and <a href="../docs/evaluation-policy.md">evaluation policy</a> are fail-closed: test evaluation is disabled and any future access requires separate review, explicit opt-in, immutable lineage, disclosure, and archival controls. This notebook validates policy without opening protected data.</div>

> **CODE CELL FUNCTION**
>
> Run the next walkthrough cell. This cell is package-backed and uses repository-relative paths.

In [ ]:
# CODE CELL FUNCTION:
# Validate the benchmark-governance policy without accessing protected test data.
#
# COMMENT MAP:
# - This cell reads only the committed benchmark policy configuration.
# - Assertions are intentional because a changed policy boundary should stop the walkthrough.
# - The assertions do not open data, compute metrics, or touch generated artifacts.

benchmark_policy = load_benchmark_policy(paths.configs / "benchmark-policy-v1.toml")

# These assertions document the repository governance boundary: routine notebooks
# cannot silently become benchmark/test-evaluation workflows.
assert benchmark_policy.protected_partition == "test"
assert benchmark_policy.test_evaluation_enabled is False
assert benchmark_policy.explicit_future_opt_in_required is True

{
    "policy": f"{benchmark_policy.policy_id}@{benchmark_policy.version}",
    "protected_partition": benchmark_policy.protected_partition,
    "test_evaluation_enabled": benchmark_policy.test_evaluation_enabled,
    "future_opt_in_required": benchmark_policy.explicit_future_opt_in_required,
}


## 11. Portfolio interpretation and limitations

The engineering result is a reproducible, configuration-driven local pipeline with explicit provenance, validation, lineage, subject-aware preparation, testable transformations, and operational evidence. It demonstrates data-engineering and platform-engineering judgment.

<div role="note" aria-label="Interpretation limitations" style="border: 3px solid #007c91; border-left-width: 8px; border-radius: 6px; background: #eefbff; color: #102a43; padding: 1rem 1.25rem; margin: 1rem 0; line-height: 1.5;"><strong style="display: block; font-size: 1.15rem; margin-bottom: 0.35rem;">Interpretation limitations</strong>This walkthrough does not establish model quality, unseen-population generalization, clinical validity, diagnostic value, medical utility, production readiness, or deployment fitness. Historical notebook metrics remain historical artifacts. Modern validation-only metrics remain development evidence. No protected test metric or supported benchmark is produced here.</div>

Continue to the [`02` gradient boosting validation example](./02-high-performing-gradient-boosting-validation.ipynb) only after Step 0 has generated the required local artifacts.

## Appendix: Version history and change log

These maintenance notes are preserved for auditability without interrupting the narrative above.

| Version | Change |
|---:|---|
| 1.10 | Fixed the local-checkout kernel guard so it accepts uv's symlinked `.venv` interpreter: it now compares the interpreter path lexically like Step 0 instead of resolving the symlink to the base Python, which had falsely rejected the correctly selected kernel. |
| 1.9 | Explained the purpose of the configured split and training seeds where the walkthrough surfaces them: fixed seeds make partition membership and the baseline fit reproducible across reruns. |
| 1.8 | Restored the supported workflow to one local VS Code/Jupyter checkout and locked project kernel. The opening cell now confirms local continuity without copying artifacts; optional web-runtime integration is deferred to Issue #200. |
| 1.7 | Superseded by 1.8: explored a cross-runtime continuity handoff during review; that unvalidated path is not part of the supported notebook workflow. |
| 1.6 | Added a conversational overview, qualified first-read/rerun timing, and compact jump links; moved this history to an appendix; and made the configured channel name and excluded record IDs visible in the windowing summary. |
| 1.5 | Added one concise start/completion pair with a qualified time expectation around optional local run-evidence discovery. Near-instant narrative/configuration cells remain quiet. |
| 1.4 | Replaced the hand-authored lineage SVG with the maintainer-approved Graphviz design and linked its tracked source and reproducible diagram workflow. |
| 1.3 | Added repository-relative document and workflow links plus accessible callout panels for prerequisites, use boundaries, benchmark governance, and interpretation limits. |
| 1.2 | Restored strict dependency behavior for the public workflow. Step 1 now requires Step 0 to complete artifact generation and verifies dataset, split, split-quality, and run-manifest artifacts before continuing. |
| 1.1 | Converts Step 0 readiness from a hard failure into a non-throwing status report so the narrative walkthrough can still execute when Step 0 ends in a controlled PIPE-006 blocked state. Adds visible internal comments to every Python cell, documents mutation boundaries, and keeps generated-artifact evidence inspection optional. |
| 1.0 | Initial public narrative walkthrough for the supported modernization lifecycle. |
